# AENZO LABS — Telegram Login (Android-friendly)
Run the code cell below. It uses simple private prompts instead of widgets. Enter API ID, API Hash and phone number only in this Colab notebook. Never share the OTP, 2FA password or session string in chat or commit them to GitHub.

In [ ]:
!pip -q install telethon
from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import SessionPasswordNeededError
from getpass import getpass

api_id = int(input('Telegram API ID: ').strip())
api_hash = getpass('Telegram API HASH (hidden): ').strip()
phone = input('Phone number with country code (example +880...): ').strip()

client = TelegramClient(StringSession(), api_id, api_hash)
await client.connect()
if not await client.is_user_authorized():
    sent = await client.send_code_request(phone)
    code = getpass('Telegram login code (hidden): ').strip()
    try:
        await client.sign_in(phone=phone, code=code, phone_code_hash=sent.phone_code_hash)
    except SessionPasswordNeededError:
        twofa = getpass('Telegram 2-step verification password (hidden): ')
        await client.sign_in(password=twofa)

print('\nLOGIN SUCCESSFUL. Copy the session string below directly into GitHub Actions secret TELEGRAM_SESSION_STRING. Do not send it to anyone.\n')
print(client.session.save())
await client.disconnect()
